In [69]:
import pandas as pd
from pandas.core.groupby import groupby


def profile_df(df, name):
    print(f"\n{'=' * 40}")
    print(f"DATASET: {name}")
    print(f"{'=' * 40}")

    print(f"\nShape:\n"
          f"{df.shape}")

    print(f"\nColumns:\n"
          f"{df.columns.to_list()}")

    print(f"\nData type:\n"
          f"{df.dtypes}")

    print(f"\nMissing values:\n"
          f"{df.isnull().sum()}")

    print(f"\nDuplicated values:\n"
          f"{df.duplicated().sum()}")

    print(f"\nFirst 5 rows:\n"
          f"{df.head()}")


inventory_df = pd.read_csv("../data/raw/inventory_of_generation_2025.csv", sep="\t")
domestic_df = pd.read_csv("../data/raw/monthly_domestic_values_2025.csv", sep="\t")
hourly_load_df = pd.read_csv("../data/raw/monthly_hourly_load_values_2025.csv", sep="\t")
physical_energy_df = pd.read_csv("../data/raw/physical_energy_power_flows_2025.csv", sep="\t")


profile_df(hourly_load_df, "Monthly Hourly Load Values")
profile_df(physical_energy_df, "Physical Energy Power Flows")



DATASET: Monthly Hourly Load Values

Shape:
(316883, 11)

Columns:
['MeasureItem', 'DateUTC', 'DateShort', 'TimeFrom', 'TimeTo', 'CountryCode', 'Cov_ratio', 'Value', 'Value_ScaleTo100', 'CreateDate', 'UpdateDate']

Data type:
MeasureItem             str
DateUTC                 str
DateShort               str
TimeFrom                str
TimeTo                  str
CountryCode             str
Cov_ratio             int64
Value               float64
Value_ScaleTo100    float64
CreateDate              str
UpdateDate              str
dtype: object

Missing values:
MeasureItem         0
DateUTC             0
DateShort           0
TimeFrom            0
TimeTo              0
CountryCode         0
Cov_ratio           0
Value               0
Value_ScaleTo100    0
CreateDate          0
UpdateDate          0
dtype: int64

Duplicated values:
0

First 5 rows:
                  MeasureItem           DateUTC   DateShort TimeFrom TimeTo  \
0  Monthly Hourly Load Values  20-08-2025 21:00  20-08-2025    

# 1. Invesory of Generation

## 1.1 Overview

* How many records are in the dataset?
* What columns are available?
* What are the data types?
* Are there missing values?
* Are there duplicate records?

In [70]:
profile_df(inventory_df, "Inventory of Generation")


DATASET: Inventory of Generation

Shape:
(211, 11)

Columns:
['MeasureItem', 'MeasureItemCategoryID', 'MeasureItemID', 'Category', 'Country', 'Year', 'ProvidedValue', 'ProvidedValueCode', 'NumberOfUnits', 'CreationDate', 'UpdateDate']

Data type:
MeasureItem                  str
MeasureItemCategoryID      int64
MeasureItemID                str
Category                     str
Country                      str
Year                       int64
ProvidedValue            float64
ProvidedValueCode        float64
NumberOfUnits              int64
CreationDate                 str
UpdateDate                   str
dtype: object

Missing values:
MeasureItem                0
MeasureItemCategoryID      0
MeasureItemID              0
Category                   0
Country                    0
Year                       0
ProvidedValue              0
ProvidedValueCode        211
NumberOfUnits              0
CreationDate               0
UpdateDate                 0
dtype: int64

Duplicated values:
0

Fir

## 1.2 Categories and Countries

* What categories are present?
* How frequently does each category occur?
* How many countries are represented?
* How many records does each country have?
* Which categories are available for each country?
* Is there more than one record for the same country and generation category?
* Why does every country contain an Error category?

In [71]:
print(inventory_df['Category'].value_counts())

print(f"\n\n{inventory_df.groupby('Country')['Category'].value_counts()}")
# Is there more than one record for the same country and generation category?
print(f"\n\nDuplicate Country + Category combinations:\n{(inventory_df.groupby(['Country', 'Category']).size().loc[lambda x: x >=2])}")
# Countr + Category <= candidate key

print(f"\n\nThe Error category:\n{inventory_df.loc[inventory_df['Category'] == 'Error', 'Country'].value_counts()}")
# How many countries are represented in the dataset?
print(f"\n\nCountries:{inventory_df['Country'].nunique()}\n{inventory_df['Country'].unique()}")
# The dataset contains 36 countries, and each country has exactly one record with the Error category.

# How many records does each country have?
print(f"\n\nRecords:\n{inventory_df['Country'].value_counts()}")

# Which categories are available for each country?
inventory_df.groupby('Country')['Category'].agg(list)

Category
Error                  36
FF_200MW_400MW         32
FF_10MW_200MW          28
Hydro_Over_100MW       26
FF_over_400MW          23
Nuclear                15
Hydro_50MW_100MW        9
Biomass_Over_1MW        9
Hydro_10MW_50MW         7
Solar_Over_1MW          7
Wind_On_Over_1MW        6
Wind_Off_Over_1MW       5
FF_LT10MW               2
Hydro_1MW_10MW          2
Waste_Over_1MW          2
Solar_LT1MW             1
GeoThermal_Over_1MW     1
Name: count, dtype: int64


Country  Category        
AL       Error               1
         Solar_Over_1MW      1
         Hydro_Over_100MW    1
         Hydro_50MW_100MW    1
AT       FF_200MW_400MW      1
                            ..
UA_IPS   Nuclear             1
XK       Wind_On_Over_1MW    1
         Hydro_10MW_50MW     1
         FF_200MW_400MW      1
         Error               1
Name: count, Length: 211, dtype: int64


Duplicate Country + Category combinations:
Series([], dtype: int64)


The Error category:
Country
AL        1
AT 

Country
AL        [Hydro_50MW_100MW, Hydro_Over_100MW, Solar_Ove...
AT        [FF_10MW_200MW, FF_200MW_400MW, FF_over_400MW,...
BA        [FF_10MW_200MW, FF_200MW_400MW, Hydro_Over_100...
BE        [Nuclear, FF_10MW_200MW, FF_200MW_400MW, FF_ov...
BG        [Nuclear, FF_10MW_200MW, FF_200MW_400MW, Hydro...
CH                       [Nuclear, Hydro_Over_100MW, Error]
CZ        [Nuclear, FF_10MW_200MW, FF_200MW_400MW, FF_ov...
DE        [FF_10MW_200MW, FF_200MW_400MW, FF_over_400MW,...
DK        [FF_10MW_200MW, FF_200MW_400MW, FF_over_400MW,...
EE                   [FF_10MW_200MW, FF_200MW_400MW, Error]
ES        [Nuclear, FF_200MW_400MW, FF_over_400MW, Hydro...
FI        [Nuclear, FF_10MW_200MW, FF_200MW_400MW, FF_ov...
FR        [Nuclear, FF_10MW_200MW, FF_200MW_400MW, FF_ov...
GB        [Nuclear, FF_LT10MW, FF_10MW_200MW, FF_200MW_4...
GE        [FF_10MW_200MW, FF_200MW_400MW, Hydro_Over_100...
GR        [FF_10MW_200MW, FF_200MW_400MW, FF_over_400MW,...
HU        [Nuclear, FF_LT10MW, F

## 1.3 ID

* What does MeasureItemID represent?
* What does MeasureItemCategoryID represent?
* Are these identifiers consistent across all records?
* Does each identifier correspond to exactly one logical entity?
* Does each MeasureItemID correspond to exactly one Category?

In [72]:
print(f"{inventory_df['MeasureItemID'].value_counts()}")
print(f"\n\n{inventory_df['MeasureItemCategoryID'].value_counts()}")
# MeasureItemID appears to identify the generation category, while MeasureItemCategoryID is constant across the dataset.
print(f"\n\n{inventory_df.groupby('MeasureItemID')['Category'].nunique()}")
# MeasureItemID 1 : 1 Category
# MeasureItemID uniquely identifies a generation category, while MeasureItemCategoryID is constant across the dataset.
print(f"\n\n{inventory_df.groupby('Category')['MeasureItemID'].nunique()}")

# Why is MeasureItemCategoryID always equal to 9?
# MeasureItemCategoryID has a constant value of 9 across all records. Its meaning should be verified against the source metadata.

MeasureItemID
Error    36
377      32
376      28
383      26
378      23
157      15
382       9
395       9
381       7
385       7
391       6
389       5
375       2
380       2
397       2
384       1
399       1
Name: count, dtype: int64


MeasureItemCategoryID
9    211
Name: count, dtype: int64


MeasureItemID
157      1
375      1
376      1
377      1
378      1
380      1
381      1
382      1
383      1
384      1
385      1
389      1
391      1
395      1
397      1
399      1
Error    1
Name: Category, dtype: int64


Category
Biomass_Over_1MW       1
Error                  1
FF_10MW_200MW          1
FF_200MW_400MW         1
FF_LT10MW              1
FF_over_400MW          1
GeoThermal_Over_1MW    1
Hydro_10MW_50MW        1
Hydro_1MW_10MW         1
Hydro_50MW_100MW       1
Hydro_Over_100MW       1
Nuclear                1
Solar_LT1MW            1
Solar_Over_1MW         1
Waste_Over_1MW         1
Wind_Off_Over_1MW      1
Wind_On_Over_1MW       1
Name: MeasureItemID, dtype: i

## 1.4 Time

* Does the dataset contain only 2025?

In [73]:
print(f"{inventory_df['Year'].value_counts()}")
# All records refer to the year 2025.

Year
2025    211
Name: count, dtype: int64


## 1.5 Values

* What are the minimum, maximum, and average values?
* Are there zero or negative values?
* Is ProvidedValueCode populated?
* If it is empty, can it be safely excluded from the analytical layer?

In [74]:
print(f"\n{'=' * 40}"
      f"\nProvidedValue"
      f"\n{'=' * 40}"
      f"\nMinimum: {inventory_df['ProvidedValue'].min()} "
      f"\nMaximum: {inventory_df['ProvidedValue'].max()} "
      f"\nAverage: {inventory_df['ProvidedValue'].mean():.2f}")

print(f"\n{'=' * 40}\n"
      f"{inventory_df['ProvidedValueCode'].isnull().value_counts()}")

print(f"\n{'=' * 40}"
      f"\nNumberOfUnits"
      f"\n{'=' * 40}"
      f"\nMinimum: {inventory_df['NumberOfUnits'].min()} "
      f"\nMaximum: {inventory_df['NumberOfUnits'].max()} "
      f"\nAverage: {inventory_df['NumberOfUnits'].mean():.2f}")

# NumberOfUnits ranges from 1 to 795, with an average of 35.31, and is generally much smaller than ProvidedValue.
# ProvidedValueCode is missing for all 211 records. Since the column contains no information in this dataset, it can be excluded from the analytical layer.
# The value-related fields contain no obvious data quality issues.


ProvidedValue
Minimum: 2.0 
Maximum: 208221.0 
Average: 9609.58

ProvidedValueCode
True    211
Name: count, dtype: int64

NumberOfUnits
Minimum: 1 
Maximum: 795 
Average: 35.31


## 1.6 Conclusions

* Dataset contains 211 records.
* Grain: one record per Country + Category for 2025.
* Main dimensions: Country, Category.
* Main measures: ProvidedValue, NumberOfUnits.
* Retained: Country, Category, Year, MeasureItemID, MeasureItemCategoryID, ProvidedValue, and NumberOfUnits.
* Remove: ProvidedValueCode (empty), CreationDate, UpdateDate.
* Issues: Error is a systematic category present for every country and should not be removed without understanding its source meaning. MeasureItemCategoryID is constant (9) and its exact meaning should be verified from source metadata.

# 2. Monthly Domestic Values

## 2.1 Overview

* How many records are in the dataset?
* What columns are available?
* What are the data types?
* Are there missing values?
* Are there duplicate records?

In [75]:
profile_df(domestic_df, "Monthly Domestic Values")


DATASET: Monthly Domestic Values

Shape:
(5576, 9)

Columns:
['MeasureItem', 'Month', 'Year', 'Category', 'Area', 'MeasureItemID', 'Representativity', 'ProvidedValue', 'ProvidedValueCode']

Data type:
MeasureItem              str
Month                  int64
Year                   int64
Category                 str
Area                     str
MeasureItemID            str
Representativity       int64
ProvidedValue        float64
ProvidedValueCode    float64
dtype: object

Missing values:
MeasureItem             0
Month                   0
Year                    0
Category                0
Area                    0
MeasureItemID           0
Representativity        0
ProvidedValue           0
ProvidedValueCode    5576
dtype: int64

Duplicated values:
0

First 5 rows:
               MeasureItem  Month  Year Category Area MeasureItemID  \
0  Monthly Domestic Values      1  2025   Export   AL            67   
1  Monthly Domestic Values      2  2025   Export   AL            67   
2  Monthl

## 2.2 Categories and Areas

* What domestic categories are present?
* How many areas are represented?
* How many records does each category contain?
* How many records does each area contain?

In [76]:
print(f"\n{'=' * 40}"
      f"\n{domestic_df['Category'].value_counts()}")

print(f"\n{'=' * 40}"
      f"\n{domestic_df['Category'].nunique()}\n{domestic_df['Category'].unique()}")

print(f"\n{'=' * 40}"
      f"\n{domestic_df['Area'].value_counts()}")

print(f"\n{'=' * 40}"
      f"\n{domestic_df['Area'].nunique()}\n{domestic_df['Area'].unique()}")

print(f"\n{'=' * 40}"
      f"\n{domestic_df.groupby('Area')['Category'].value_counts()}"
      f"\n{'=' * 40}"
      f"\n{domestic_df.groupby('Category')['Area'].value_counts()}")

# Does each Area + Category contain data for all 12 months?
months_per_group = (domestic_df.groupby(['Area', 'Category'])['Month'].nunique())

missing_months = (domestic_df.groupby(['Area', 'Category'])['Month'].apply(lambda x: sorted(set(range(1,13)) - set(x))))

print(f"\n{'=' * 40}"
      f"\n{months_per_group[months_per_group != 12]}"
      f"\n{'=' * 40}"
      f"\n{missing_months[missing_months.apply(len) > 0]}")

# Most Area + Category combinations contain all 12 months.
# 34 combinations have incomplete monthly coverage.
# Missing months vary by area and category, so they should be retained rather than treated as one general data quality issue.


Category
Import                                            501
Export                                            496
Wind Onshore                                      419
Fossil Gas                                        372
Solar                                             364
Hydro Run-of-river and poundage                   336
Biomass                                           311
Hydro Water Reservoir                             276
Hydro Pumped Storage                              240
Other                                             240
Fossil Oil                                        239
Fossil Hard coal                                  222
Waste                                             216
Fossil Brown coal/Lignite                         180
Consumption of Hydro Pumped Storage               160
Nuclear                                           156
Other renewable                                   151
Wind Offshore                                     101
Fossil Coal-derive

## 2.3 ID

## 2.4 Time

## 2.5 Values

## 2.6 Conclusions

* Dataset contains  records.
* Grain:
* Main dimensions:
* Main measures:
* Retained:
* Remove:
* Issues: